# Food-101 and Nutrition5k Data Exploration

This notebook checks Food-101 class balance and sample images, then summarizes Nutrition5k dish nutrition values and flags unusual records for manual review. Dataset paths are read from `ml/configs/base.yaml`. Outlier flags are screening heuristics only; they do not alter the data.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import math

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from PIL import Image


cwd = Path.cwd().resolve()
ml_dir = None
for candidate in (cwd, *cwd.parents):
    if (candidate / "configs" / "base.yaml").is_file():
        ml_dir = candidate
        break
    if (candidate / "ml" / "configs" / "base.yaml").is_file():
        ml_dir = candidate / "ml"
        break
if ml_dir is None:
    raise FileNotFoundError("Could not find ml/configs/base.yaml from the current directory or its parents.")

with (ml_dir / "configs" / "base.yaml").open(encoding="utf-8") as config_file:
    config = yaml.safe_load(config_file)


def ml_path(config_value):
    path = Path(config_value)
    return path if path.is_absolute() else ml_dir / path


food101_dir = ml_path(config["dataset"]["food101_dir"]) / "food-101"
nutrition_csv = ml_path(config["dataset"]["nutrition5k_clean_csv"])
print(f"ML directory: {ml_dir}")
print(f"Food-101 extracted directory: {food101_dir}")
print(f"Nutrition5k clean CSV: {nutrition_csv}")

## Food-101 Class Balance

The torchvision dataset stores its class list and split manifests under `food-101/meta`. Counts below come from the train and test manifests, grouped by class.

In [ ]:
metadata_dir = food101_dir / "meta"
classes_file = metadata_dir / "classes.txt"
train_file = metadata_dir / "train.txt"
test_file = metadata_dir / "test.txt"
missing = [path for path in (classes_file, train_file, test_file) if not path.is_file()]
if missing:
    missing_list = "\n".join(f"- {path}" for path in missing)
    raise FileNotFoundError(
        "Food-101 files are missing:\n"
        f"{missing_list}\n"
        "Run `python ml/data/download_food101.py` from the repository root first."
    )

with classes_file.open(encoding="utf-8") as file:
    food_classes = [line.strip() for line in file if line.strip()]


def count_split_classes(manifest_path):
    counts = Counter()
    with manifest_path.open(encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            image_id = line.strip()
            if not image_id:
                continue
            class_name = image_id.split("/", maxsplit=1)[0]
            if class_name not in food_classes:
                raise ValueError(f"Unknown class {class_name!r} in {manifest_path}:{line_number}")
            counts[class_name] += 1
    return counts


train_counts = count_split_classes(train_file)
test_counts = count_split_classes(test_file)
class_counts = pd.DataFrame(
    {
        "train_images": [train_counts[name] for name in food_classes],
        "test_images": [test_counts[name] for name in food_classes],
    },
    index=food_classes,
)
class_counts["total_images"] = class_counts["train_images"] + class_counts["test_images"]
class_counts.loc["TOTAL"] = class_counts.sum(axis=0)
display(class_counts)
print(f"Food-101 classes: {len(food_classes)}")

## Food-101 Sample Images

Display one available training image from up to 12 classes to visually inspect the extracted images and labels.

In [ ]:
sample_ids = {}
with train_file.open(encoding="utf-8") as file:
    for line in file:
        image_id = line.strip()
        if image_id:
            class_name = image_id.split("/", maxsplit=1)[0]
            sample_ids.setdefault(class_name, image_id)

sample_classes = [name for name in food_classes if name in sample_ids][:12]
if not sample_classes:
    raise ValueError("No sample images were listed in the Food-101 training manifest.")

columns = 4
rows = math.ceil(len(sample_classes) / columns)
fig, axes = plt.subplots(rows, columns, figsize=(14, 3.5 * rows))
axes = list(axes.flat) if hasattr(axes, "flat") else [axes]
for axis, class_name in zip(axes, sample_classes):
    image_stem = sample_ids[class_name].split("/", maxsplit=1)[1]
    image_path = food101_dir / "images" / class_name / f"{image_stem}.jpg"
    if not image_path.is_file():
        raise FileNotFoundError(f"Food-101 image listed in train.txt was not found: {image_path}")
    with Image.open(image_path) as image:
        axis.imshow(image.convert("RGB"))
    axis.set_title(class_name.replace("_", " ").title())
    axis.axis("off")
for axis in axes[len(sample_classes):]:
    axis.axis("off")
fig.tight_layout()
plt.show()

## Nutrition5k Summary Statistics

The cleaner writes dish-level labels to the CSV configured by `dataset.nutrition5k_clean_csv`. Summary statistics below are computed without modifying the source values.

In [ ]:
if not nutrition_csv.is_file():
    raise FileNotFoundError(
        f"Clean Nutrition5k CSV not found: {nutrition_csv}\n"
        "Place the official dish metadata under the configured raw-data directory, "
        "then run `python ml/data/prepare_nutrition5k.py`. Download instructions: "
        "https://github.com/google-research-datasets/Nutrition5k#download-data"
    )

nutrition = pd.read_csv(nutrition_csv)
required_columns = [
    "dish_id",
    "calories",
    "protein_g",
    "carbohydrates_g",
    "fat_g",
]
missing_columns = [column for column in required_columns if column not in nutrition.columns]
if missing_columns:
    raise ValueError(f"Nutrition CSV is missing required columns: {missing_columns}")

nutrition_fields = ["calories", "protein_g", "carbohydrates_g", "fat_g"]
for column in nutrition_fields:
    nutrition[column] = pd.to_numeric(nutrition[column], errors="coerce")
invalid_counts = nutrition[nutrition_fields].isna().sum()
if invalid_counts.any():
    raise ValueError(f"Nutrition CSV has missing or non-numeric values:\n{invalid_counts}")

summary = nutrition[nutrition_fields].agg(["mean", "median", "min", "max"]).T
summary.index = ["Calories (kcal)", "Protein (g)", "Carbohydrates (g)", "Fat (g)"]
summary = summary.rename(columns={"min": "minimum", "max": "maximum"})
print(f"Nutrition5k dish records: {len(nutrition):,}")
display(summary)

## Calorie Distribution

A histogram shows the range and shape of dish-level calorie estimates in the cleaned metadata.

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(nutrition["calories"], bins=40, color="#d97745", edgecolor="white")
plt.title("Nutrition5k Dish Calories")
plt.xlabel("Calories (kcal)")
plt.ylabel("Number of dishes")
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

## Candidate Outliers for Manual Review

The thresholds below are screening heuristics, not corrections or ground truth. Tukey fences flag values outside 1.5 times the interquartile range. Review the source dish records before making any data-quality decisions; this notebook does not remove or change any rows.

In [ ]:
CALORIE_LOW_REVIEW_KCAL = 50
CALORIE_HIGH_REVIEW_KCAL = 1500
outlier_reasons = defaultdict(set)

low_calorie = nutrition.index[nutrition["calories"] < CALORIE_LOW_REVIEW_KCAL]
high_calorie = nutrition.index[nutrition["calories"] > CALORIE_HIGH_REVIEW_KCAL]
for row_index in low_calorie:
    outlier_reasons[row_index].add(f"calories below {CALORIE_LOW_REVIEW_KCAL} kcal")
for row_index in high_calorie:
    outlier_reasons[row_index].add(f"calories above {CALORIE_HIGH_REVIEW_KCAL} kcal")

for field in nutrition_fields:
    first_quartile = nutrition[field].quantile(0.25)
    third_quartile = nutrition[field].quantile(0.75)
    iqr = third_quartile - first_quartile
    lower_fence = first_quartile - 1.5 * iqr
    upper_fence = third_quartile + 1.5 * iqr
    outside_fences = nutrition.index[
        (nutrition[field] < lower_fence) | (nutrition[field] > upper_fence)
    ]
    for row_index in outside_fences:
        outlier_reasons[row_index].add(f"{field} outside Tukey fences")

if outlier_reasons:
    candidate_indices = sorted(outlier_reasons)
    outlier_review = nutrition.loc[candidate_indices, required_columns].copy()
    outlier_review["review_reason"] = [
        "; ".join(sorted(outlier_reasons[index])) for index in candidate_indices
    ]
    print(f"Candidate dishes for manual review: {len(outlier_review):,}")
    display(outlier_review.sort_values("calories", ascending=False).reset_index(drop=True))
else:
    print("No records matched the configured screening heuristics.")